# 模块R · R6：研究伦理与AI治理

> **所属**：AI原生化商业博士 · 模块R 博士研究方法论 · R6（模块R收官单元）
> **上机练习**：用 pydantic + pandas 实现 Belmont Report 伦理审查清单 + NIST AI RMF 研究伦理映射 + 红队测试伦理验证
> **真实库**：pydantic（伦理审查schema）、pandas（审查结果分析）、garak/PyRIT（AI安全测试伦理验证概念）
> **真实数据**：基于 OECD AI Incidents Monitor 真实事件类型构建的AI研究案例集
> **v5.0 哲学**：真实即严谨，练习即掌握


## 上机概览

本 notebook 从**研究伦理视角**（区别于技能2 Day1的企业治理视角、技能5 Day4的安全防护视角）实现三个核心工具：

1. **Belmont Report 伦理审查清单**：用 pydantic 定义三原则（尊重个人/善行/公平正义）6个审查项的schema，对AI研究案例做IRB式伦理审查
2. **NIST AI RMF 研究伦理映射 + EU AI Act 研究合规**：从研究伦理视角映射NIST四步循环，判定EU AI Act风险等级
3. **AI红队测试伦理验证 + 天道推演**：用 garak/PyRIT 概念做AI研究的伦理验证（红队测试作为善行原则的履行），用天道推演预判伦理风险路径

**6个TODO**：
- TODO1：Belmont Report 伦理审查清单 schema（pydantic）
- TODO2：真实AI研究案例集（OECD AI Incidents 事件类型）
- TODO3：IRB 伦理审查评分器（Belmont三原则评分）
- TODO4：NIST AI RMF 研究伦理映射 + EU AI Act 研究合规判定
- TODO5：pandas 伦理审查结果分析（案例×原则热力图）
- TODO6：AI红队测试伦理验证（garak/PyRIT）+ 天道推演预判伦理风险路径


## 环境准备

导入真实库：pydantic（数据验证）、pandas（结果分析）。garak/PyRIT 较重，本单元用其真实概念做轻量模拟，data/README.md 说明安装用法。


In [1]:
from pydantic import BaseModel, Field
from enum import Enum
from typing import Optional
import pandas as pd

print('pydantic + pandas loaded successfully')
print(f'pydantic version: {__import__("pydantic").__version__}')
print(f'pandas version: {pd.__version__}')


pydantic + pandas loaded successfully
pydantic version: 2.12.5
pandas version: 2.2.2


## TODO1：Belmont Report 伦理审查清单 Schema

Belmont Report（1979）确立了三条研究伦理核心原则，至今仍是IRB（Institutional Review Board）审查的基础：

| 原则 | 英文 | 核心要求 |
|:----:|:----:|---------|
| **尊重个人** | Respect for Persons | 知情同意（informed consent）+ 自主决策保护 |
| **善行** | Beneficence | 最大化收益、最小化伤害（风险-收益评估） |
| **公平正义** | Justice | 研究负担和收益的公平分配 |

用 pydantic 定义伦理审查清单的schema：ComplianceStatus枚举 + EthicsChecklistItem模型 + 6个真实审查项列表。


In [2]:
# Belmont Report (1979) 三原则 - IRB伦理审查的基础
class BelmontPrinciple(str, Enum):
    RESPECT_FOR_PERSONS = "尊重个人"
    BENEFICENCE = "善行"
    JUSTICE = "公平正义"

class ReviewStatus(str, Enum):
    NOT_ASSESSED = "未评估"
    NONCOMPLIANT = "不合规"
    PARTIAL = "部分合规"
    COMPLIANT = "合规"

class EthicsChecklistItem(BaseModel):
    principle: BelmontPrinciple
    item_id: str
    description: str
    status: ReviewStatus = ReviewStatus.NOT_ASSESSED
    score: float = Field(ge=0, le=100, default=0.0)
    evidence: str = ""

# Belmont Report 三原则 6 个审查项（基于真实原则，Nuremberg Code 1947 -> Helsinki 1964 -> Belmont 1979）
BELMONT_CHECKLIST = [
    EthicsChecklistItem(principle=BelmontPrinciple.RESPECT_FOR_PERSONS, item_id="R1",
        description="知情同意：参与者充分知情后自愿同意（autonomy + informed consent）"),
    EthicsChecklistItem(principle=BelmontPrinciple.RESPECT_FOR_PERSONS, item_id="R2",
        description="自主决策保护：对无法自主决策群体（未成年人/认知障碍）的额外保护"),
    EthicsChecklistItem(principle=BelmontPrinciple.BENEFICENCE, item_id="B1",
        description="风险-收益评估：最大化收益、最小化伤害（maximize benefits, minimize harms）"),
    EthicsChecklistItem(principle=BelmontPrinciple.BENEFICENCE, item_id="B2",
        description="隐私保护：数据脱敏与差分隐私（Differential Privacy, Dwork & Roth 2014）"),
    EthicsChecklistItem(principle=BelmontPrinciple.JUSTICE, item_id="J1",
        description="负担公平分配：弱势群体不过度承担研究风险"),
    EthicsChecklistItem(principle=BelmontPrinciple.JUSTICE, item_id="J2",
        description="收益公平分配：优势群体不独享研究收益"),
]

print(f"Belmont Report 伦理审查清单：{len(BELMONT_CHECKLIST)} 项，覆盖 {len(set(i.principle for i in BELMONT_CHECKLIST))} 原则")
for item in BELMONT_CHECKLIST:
    print(f"  [{item.item_id}] {item.principle.value:6s} | {item.description[:35]}...")


Belmont Report 伦理审查清单：6 项，覆盖 3 原则
  [R1] 尊重个人   | 知情同意：参与者充分知情后自愿同意（autonomy + inform...
  [R2] 尊重个人   | 自主决策保护：对无法自主决策群体（未成年人/认知障碍）的额外保护...
  [B1] 善行     | 风险-收益评估：最大化收益、最小化伤害（maximize benefi...
  [B2] 善行     | 隐私保护：数据脱敏与差分隐私（Differential Privacy...
  [J1] 公平正义   | 负担公平分配：弱势群体不过度承担研究风险...
  [J2] 公平正义   | 收益公平分配：优势群体不独享研究收益...


## TODO2：真实AI研究案例集（OECD AI Incidents Monitor）

构建8个AI研究案例，基于 **OECD AI Incidents Monitor**（https://oecd.ai/en/incidents-overview）的真实AI事件类型。每个案例代表一类真实AI研究场景，需做伦理审查。

用 pydantic 定义 ResearchCase 模型，包含伦理审查所需的属性：是否涉及人类数据、是否使用敏感属性、是否有知情同意、潜在伤害严重度、是否影响弱势群体、是否自主决策。


In [3]:
class ResearchCase(BaseModel):
    case_id: str
    title: str
    domain: str  # marketing/hr/finance/healthcare/security
    data_involves_human: bool
    uses_sensitive_attributes: bool
    has_informed_consent: bool
    potential_harm_severity: str  # low/medium/high/severe
    affects_vulnerable_groups: bool
    is_autonomous_decision: bool
    oecd_event_type: str  # 真实 OECD AI Incidents 事件类型

# 基于 OECD AI Incidents Monitor (https://oecd.ai/en/incidents-overview) 真实事件类型构建
RESEARCH_CASES = [
    ResearchCase(case_id="C001", title="AI个性化推荐系统用户行为研究", domain="marketing",
        data_involves_human=True, uses_sensitive_attributes=False, has_informed_consent=True,
        potential_harm_severity="low", affects_vulnerable_groups=False, is_autonomous_decision=True,
        oecd_event_type="算法偏见/信息茧房"),
    ResearchCase(case_id="C002", title="AI自动营销文案生成与A/B测试", domain="marketing",
        data_involves_human=True, uses_sensitive_attributes=False, has_informed_consent=False,
        potential_harm_severity="medium", affects_vulnerable_groups=False, is_autonomous_decision=True,
        oecd_event_type="虚假宣传/误导信息"),
    ResearchCase(case_id="C003", title="AI动态定价系统用户支付意愿研究", domain="marketing",
        data_involves_human=True, uses_sensitive_attributes=True, has_informed_consent=False,
        potential_harm_severity="high", affects_vulnerable_groups=True, is_autonomous_decision=True,
        oecd_event_type="价格歧视/算法共谋"),
    ResearchCase(case_id="C004", title="AI客服聊天机器人用户交互研究", domain="marketing",
        data_involves_human=True, uses_sensitive_attributes=False, has_informed_consent=True,
        potential_harm_severity="medium", affects_vulnerable_groups=False, is_autonomous_decision=True,
        oecd_event_type="有害建议/误导信息"),
    ResearchCase(case_id="C005", title="AI简历筛选系统招聘有效性研究", domain="hr",
        data_involves_human=True, uses_sensitive_attributes=True, has_informed_consent=False,
        potential_harm_severity="high", affects_vulnerable_groups=True, is_autonomous_decision=True,
        oecd_event_type="就业歧视"),
    ResearchCase(case_id="C006", title="AI信用评分模型信贷风险评估研究", domain="finance",
        data_involves_human=True, uses_sensitive_attributes=True, has_informed_consent=True,
        potential_harm_severity="severe", affects_vulnerable_groups=True, is_autonomous_decision=True,
        oecd_event_type="信贷歧视"),
    ResearchCase(case_id="C007", title="AI人脸识别门禁系统安全研究", domain="security",
        data_involves_human=True, uses_sensitive_attributes=True, has_informed_consent=False,
        potential_harm_severity="severe", affects_vulnerable_groups=False, is_autonomous_decision=True,
        oecd_event_type="隐私侵犯/误识别"),
    ResearchCase(case_id="C008", title="AI医疗影像诊断辅助系统研究", domain="healthcare",
        data_involves_human=True, uses_sensitive_attributes=True, has_informed_consent=True,
        potential_harm_severity="severe", affects_vulnerable_groups=True, is_autonomous_decision=False,
        oecd_event_type="误诊/安全风险"),
]

print(f"AI研究案例集：{len(RESEARCH_CASES)} 个案例，覆盖 {len(set(c.domain for c in RESEARCH_CASES))} 个领域")
print(f"{'案例':6s} | {'领域':10s} | {'伤害':7s} | {'知情同意':4s} | {'敏感属性':4s} | OECD事件类型")
print("-" * 75)
for c in RESEARCH_CASES:
    consent = "是" if c.has_informed_consent else "否"
    sensitive = "是" if c.uses_sensitive_attributes else "否"
    print(f"{c.case_id:6s} | {c.domain:10s} | {c.potential_harm_severity:7s} | {consent:4s} | {sensitive:4s} | {c.oecd_event_type}")


AI研究案例集：8 个案例，覆盖 5 个领域
案例     | 领域         | 伤害      | 知情同意 | 敏感属性 | OECD事件类型
---------------------------------------------------------------------------
C001   | marketing  | low     | 是    | 否    | 算法偏见/信息茧房
C002   | marketing  | medium  | 否    | 否    | 虚假宣传/误导信息
C003   | marketing  | high    | 否    | 是    | 价格歧视/算法共谋
C004   | marketing  | medium  | 是    | 否    | 有害建议/误导信息
C005   | hr         | high    | 否    | 是    | 就业歧视
C006   | finance    | severe  | 是    | 是    | 信贷歧视
C007   | security   | severe  | 否    | 是    | 隐私侵犯/误识别
C008   | healthcare | severe  | 是    | 是    | 误诊/安全风险


## TODO3：IRB 伦理审查评分器

实现IRB式伦理审查评分器，对每个案例按Belmont三原则6个审查项逐一评分（0-100，越高越合规）：

1. `assess_checklist_item(case, item)` -> float：按原则和审查项属性评分
2. `score_to_status(score)` -> ReviewStatus：分数转合规状态
3. `irb_ethics_review(case)` -> dict：完整审查，输出综合分和伦理风险等级


In [4]:
def assess_checklist_item(case: ResearchCase, item: EthicsChecklistItem) -> float:
    """按 Belmont 原则对案例评分 (0-100, 越高越合规)"""
    score = 0.0
    p = item.principle
    if p == BelmontPrinciple.RESPECT_FOR_PERSONS:
        if item.item_id == "R1":  # 知情同意
            if not case.data_involves_human:
                score = 100.0
            elif case.has_informed_consent:
                score = 90.0
            else:
                score = 30.0  # 涉及人类数据但无知情同意
        elif item.item_id == "R2":  # 自主决策保护
            if case.affects_vulnerable_groups and not case.has_informed_consent:
                score = 20.0
            elif case.affects_vulnerable_groups:
                score = 60.0
            else:
                score = 95.0
    elif p == BelmontPrinciple.BENEFICENCE:
        if item.item_id == "B1":  # 风险-收益评估
            harm_map = {"low": 90.0, "medium": 65.0, "high": 40.0, "severe": 20.0}
            score = harm_map.get(case.potential_harm_severity, 50.0)
        elif item.item_id == "B2":  # 隐私保护
            if case.uses_sensitive_attributes and not case.has_informed_consent:
                score = 25.0
            elif case.uses_sensitive_attributes:
                score = 55.0
            else:
                score = 90.0
    elif p == BelmontPrinciple.JUSTICE:
        if item.item_id == "J1":  # 负担公平分配
            if case.affects_vulnerable_groups and case.potential_harm_severity in ("high", "severe"):
                score = 30.0
            elif case.affects_vulnerable_groups:
                score = 60.0
            else:
                score = 90.0
        elif item.item_id == "J2":  # 收益公平分配
            if case.is_autonomous_decision and case.potential_harm_severity in ("high", "severe"):
                score = 50.0
            else:
                score = 80.0
    return max(0.0, min(100.0, score))

def score_to_status(score: float) -> ReviewStatus:
    if score >= 80:
        return ReviewStatus.COMPLIANT
    if score >= 50:
        return ReviewStatus.PARTIAL
    if score > 0:
        return ReviewStatus.NONCOMPLIANT
    return ReviewStatus.NOT_ASSESSED

def irb_ethics_review(case: ResearchCase) -> dict:
    """IRB 式伦理审查：对案例按 Belmont 三原则 6 审查项逐一评分"""
    results = []
    for item in BELMONT_CHECKLIST:
        sc = assess_checklist_item(case, item)
        results.append(item.model_copy(update={"score": sc, "status": score_to_status(sc)}))
    overall = sum(r.score for r in results) / len(results)
    if overall >= 75:
        risk_level = "低风险"
    elif overall >= 50:
        risk_level = "中风险"
    else:
        risk_level = "高风险"
    return {"case": case, "items": results, "overall_score": round(overall, 1), "risk_level": risk_level}

print("=== IRB 伦理审查结果（Belmont Report 三原则）===")
for case in RESEARCH_CASES:
    review = irb_ethics_review(case)
    print(f"\n[{review['case'].case_id}] {review['case'].title}")
    print(f"  综合合规分: {review['overall_score']}/100 | 伦理风险: {review['risk_level']}")
    for r in review['items']:
        print(f"    {r.item_id} {r.principle.value:6s} | {r.status.value:5s} | {r.score:5.1f} | {r.description[:22]}")


=== IRB 伦理审查结果（Belmont Report 三原则）===

[C001] AI个性化推荐系统用户行为研究
  综合合规分: 89.2/100 | 伦理风险: 低风险
    R1 尊重个人   | 合规    |  90.0 | 知情同意：参与者充分知情后自愿同意（auto
    R2 尊重个人   | 合规    |  95.0 | 自主决策保护：对无法自主决策群体（未成年人/
    B1 善行     | 合规    |  90.0 | 风险-收益评估：最大化收益、最小化伤害（ma
    B2 善行     | 合规    |  90.0 | 隐私保护：数据脱敏与差分隐私（Differe
    J1 公平正义   | 合规    |  90.0 | 负担公平分配：弱势群体不过度承担研究风险
    J2 公平正义   | 合规    |  80.0 | 收益公平分配：优势群体不独享研究收益

[C002] AI自动营销文案生成与A/B测试
  综合合规分: 75.0/100 | 伦理风险: 低风险
    R1 尊重个人   | 不合规   |  30.0 | 知情同意：参与者充分知情后自愿同意（auto
    R2 尊重个人   | 合规    |  95.0 | 自主决策保护：对无法自主决策群体（未成年人/
    B1 善行     | 部分合规  |  65.0 | 风险-收益评估：最大化收益、最小化伤害（ma
    B2 善行     | 合规    |  90.0 | 隐私保护：数据脱敏与差分隐私（Differe
    J1 公平正义   | 合规    |  90.0 | 负担公平分配：弱势群体不过度承担研究风险
    J2 公平正义   | 合规    |  80.0 | 收益公平分配：优势群体不独享研究收益

[C003] AI动态定价系统用户支付意愿研究
  综合合规分: 32.5/100 | 伦理风险: 高风险
    R1 尊重个人   | 不合规   |  30.0 | 知情同意：参与者充分知情后自愿同意（auto
    R2 尊重个人   | 不合规   |  20.0 | 自主决策保护：对无法自主决策群体（未成年人/
    B1 善行     | 不合规   |  40.0 | 风险-收益评估：

## TODO4：NIST AI RMF 研究伦理映射 + EU AI Act 研究合规

**NIST AI RMF 研究伦理映射**（区别于技能2 Day1企业治理视角）：
- Govern：是否有IRB审批/伦理委员会监督
- Map：是否识别了人类参与者及风险
- Measure：是否有偏见/公平性/安全性度量
- Manage：是否有风险缓解与知情退出机制

**EU AI Act 研究合规判定**：按 Article 5(禁止) -> Annex III(高风险) -> Article 50(有限风险) -> 最小风险 判定。


In [5]:
def nist_research_ethics_mapping(case: ResearchCase) -> dict:
    """从研究伦理视角映射 NIST AI RMF 四步循环（区别于技能2 Day1 企业治理视角）"""
    mapping = {"Govern": 0.0, "Map": 0.0, "Measure": 0.0, "Manage": 0.0}
    # Govern: 是否有 IRB 审批/伦理委员会监督（知情同意 + 非自主决策=有人监督）
    mapping["Govern"] = (70.0 if case.has_informed_consent else 25.0) + (15.0 if not case.is_autonomous_decision else 0.0)
    # Map: 是否识别了人类参与者及风险
    mapping["Map"] = (60.0 if not case.uses_sensitive_attributes else 40.0) + (20.0 if case.has_informed_consent else 0.0)
    # Measure: 是否有偏见/公平性/安全性度量
    measure_boost = {"low": 20.0, "medium": 10.0, "high": 0.0, "severe": -10.0}
    mapping["Measure"] = 50.0 + measure_boost.get(case.potential_harm_severity, 0.0) + (15.0 if not case.uses_sensitive_attributes else 0.0)
    # Manage: 是否有风险缓解与知情退出机制
    mapping["Manage"] = (60.0 if case.has_informed_consent else 30.0) + (20.0 if not case.affects_vulnerable_groups else 0.0)
    for k in mapping:
        mapping[k] = round(min(100.0, max(0.0, mapping[k])), 1)
    return mapping

def classify_eu_ai_act_research(case: ResearchCase) -> tuple:
    """从研究合规视角判定 EU AI Act 风险等级
    判定顺序: Article 5(禁止) -> Annex III(高风险) -> Article 50(有限风险) -> 最小风险
    """
    if case.uses_sensitive_attributes and case.affects_vulnerable_groups and not case.has_informed_consent:
        return ("禁止", "Article 5: 针对弱势群体的剥削性处理 + 无知情同意")
    if case.potential_harm_severity in ("high", "severe") and case.uses_sensitive_attributes:
        return ("高风险", "Annex III: 对基本权利有重大影响 + 敏感属性处理")
    if case.is_autonomous_decision and case.data_involves_human:
        return ("有限风险", "Article 50: AI交互/自主决策需透明度义务")
    return ("最小风险", "对基本权利影响微小")

print("=== NIST AI RMF 研究伦理映射 + EU AI Act 研究合规 ===")
for case in RESEARCH_CASES:
    nist = nist_research_ethics_mapping(case)
    eu_level, eu_basis = classify_eu_ai_act_research(case)
    print(f"\n[{case.case_id}] {case.title}")
    print(f"  NIST(研究视角): Govern={nist['Govern']:5.1f} Map={nist['Map']:5.1f} Measure={nist['Measure']:5.1f} Manage={nist['Manage']:5.1f}")
    print(f"  EU AI Act: {eu_level} | {eu_basis}")


=== NIST AI RMF 研究伦理映射 + EU AI Act 研究合规 ===

[C001] AI个性化推荐系统用户行为研究
  NIST(研究视角): Govern= 70.0 Map= 80.0 Measure= 85.0 Manage= 80.0
  EU AI Act: 有限风险 | Article 50: AI交互/自主决策需透明度义务

[C002] AI自动营销文案生成与A/B测试
  NIST(研究视角): Govern= 25.0 Map= 60.0 Measure= 75.0 Manage= 50.0
  EU AI Act: 有限风险 | Article 50: AI交互/自主决策需透明度义务

[C003] AI动态定价系统用户支付意愿研究
  NIST(研究视角): Govern= 25.0 Map= 40.0 Measure= 50.0 Manage= 30.0
  EU AI Act: 禁止 | Article 5: 针对弱势群体的剥削性处理 + 无知情同意

[C004] AI客服聊天机器人用户交互研究
  NIST(研究视角): Govern= 70.0 Map= 80.0 Measure= 75.0 Manage= 80.0
  EU AI Act: 有限风险 | Article 50: AI交互/自主决策需透明度义务

[C005] AI简历筛选系统招聘有效性研究
  NIST(研究视角): Govern= 25.0 Map= 40.0 Measure= 50.0 Manage= 30.0
  EU AI Act: 禁止 | Article 5: 针对弱势群体的剥削性处理 + 无知情同意

[C006] AI信用评分模型信贷风险评估研究
  NIST(研究视角): Govern= 70.0 Map= 60.0 Measure= 40.0 Manage= 60.0
  EU AI Act: 高风险 | Annex III: 对基本权利有重大影响 + 敏感属性处理

[C007] AI人脸识别门禁系统安全研究
  NIST(研究视角): Govern= 25.0 Map= 40.0 Measure= 40.0 Manage= 50.0
  EU AI Act: 高风险 | Annex III: 对基本权利有重大影响 + 敏

## TODO5：pandas 伦理审查结果分析

用 pandas 将IRB审查结果转为DataFrame，构建"案例×原则"伦理合规热力图，识别合规短板（最弱原则/最弱案例）。


In [6]:
def build_ethics_heatmap(cases: list) -> tuple:
    """构建案例 x 原则 伦理合规热力图"""
    rows = []
    for case in cases:
        review = irb_ethics_review(case)
        by_principle = {}
        for r in review['items']:
            by_principle.setdefault(r.principle.value, []).append(r.score)
        for p_name, scores in by_principle.items():
            rows.append({
                "case_id": case.case_id,
                "title": case.title,
                "domain": case.domain,
                "principle": p_name,
                "avg_score": round(sum(scores) / len(scores), 1),
                "overall": review['overall_score'],
                "risk_level": review['risk_level'],
            })
    detail_df = pd.DataFrame(rows)
    pivot_df = detail_df.pivot_table(values="avg_score", index="case_id", columns="principle", aggfunc="mean")
    return detail_df, pivot_df

detail_df, pivot_df = build_ethics_heatmap(RESEARCH_CASES)

print("=== 伦理审查明细 (案例 x 原则) ===")
print(detail_df[["case_id", "domain", "principle", "avg_score", "overall", "risk_level"]].to_string(index=False))

print("\n=== 案例 x 原则 合规热力图 ===")
print(pivot_df.to_string())

print("\n=== 各原则合规短板分析 ===")
for p in pivot_df.columns:
    weakest = pivot_df[p].idxmin()
    print(f"  {p:6s}: 平均 {pivot_df[p].mean():5.1f} | 最弱案例={weakest}({pivot_df[p].min():5.1f})")

print("\n=== 各领域平均伦理风险 ===")
domain_risk = detail_df.groupby("domain")["overall"].mean().sort_values()
print(domain_risk.to_string())


=== 伦理审查明细 (案例 x 原则) ===
case_id     domain principle  avg_score  overall risk_level
   C001  marketing      尊重个人       92.5     89.2        低风险
   C001  marketing        善行       90.0     89.2        低风险
   C001  marketing      公平正义       85.0     89.2        低风险
   C002  marketing      尊重个人       62.5     75.0        低风险
   C002  marketing        善行       77.5     75.0        低风险
   C002  marketing      公平正义       85.0     75.0        低风险
   C003  marketing      尊重个人       25.0     32.5        高风险
   C003  marketing        善行       32.5     32.5        高风险
   C003  marketing      公平正义       40.0     32.5        高风险
   C004  marketing      尊重个人       92.5     85.0        低风险
   C004  marketing        善行       77.5     85.0        低风险
   C004  marketing      公平正义       85.0     85.0        低风险
   C005         hr      尊重个人       25.0     32.5        高风险
   C005         hr        善行       32.5     32.5        高风险
   C005         hr      公平正义       40.0     32.5        高风险
   C006    fina

## TODO6：AI红队测试伦理验证 + 天道推演预判伦理风险

**AI红队测试伦理验证**：用 garak（probes）+ PyRIT（风险评分）概念，将红队测试作为Belmont善行原则（最大化收益、最小化伤害）的履行手段。AI研究涉及AI系统时，红队测试是发现伤害的伦理义务。

**天道推演预判伦理风险路径**：用天道推演的沙盘模拟方法，为每个案例生成3层推演树（immediate -> near -> far），识别高杠杆干预点。连接CLAUDE.md天道推演的"因果链追踪"和"沙盘模拟"能力矩阵。


In [7]:
def redteam_ethics_validation(case: ResearchCase) -> dict:
    """用 garak/PyRIT 概念做 AI 研究的伦理验证
    红队测试作为 Belmont 善行原则（最大化收益、最小化伤害）的履行手段
    garak: probes 按领域映射漏洞类别; PyRIT: 风险评分量化
    """
    # garak probes 映射：不同领域需跑的漏洞探针（真实 garak 0.15.1 probe 类别）
    garak_probes = {
        "marketing": ["dan", "promptinject", "goodside"],   # 越狱/注入/社工
        "hr": ["dan", "encoding", "leakreplay"],             # 越狱/编码/泄露
        "finance": ["promptinject", "leakreplay", "packagehallucination"],
        "healthcare": ["dan", "snowball", "goodside"],       # 越狱/幻觉/社工
        "security": ["encoding", "promptinject", "leakreplay"],
    }
    probes = garak_probes.get(case.domain, ["dan", "promptinject"])
    # PyRIT 风险评分概念：基于案例属性估算红队发现的风险
    pyrit_risk = 0.0
    if case.uses_sensitive_attributes:
        pyrit_risk += 30.0
    if not case.has_informed_consent:
        pyrit_risk += 25.0
    if case.potential_harm_severity == "severe":
        pyrit_risk += 25.0
    elif case.potential_harm_severity == "high":
        pyrit_risk += 15.0
    if case.is_autonomous_decision:
        pyrit_risk += 10.0
    pyrit_risk = min(100.0, pyrit_risk)
    return {
        "case_id": case.case_id,
        "domain": case.domain,
        "garak_probes": probes,
        "pyrit_risk_score": round(pyrit_risk, 1),
        "ethics_basis": "红队测试履行 Belmont 善行原则（最大化收益、最小化伤害）",
    }

def tian_dao_ethics_projection(case: ResearchCase) -> dict:
    """天道推演预判 AI 研究的伦理风险路径（3层推演树）
    局势拆解 -> 因果建模 -> 沙盘展开(immediate/near/far) -> 高杠杆点识别
    """
    if case.uses_sensitive_attributes and not case.has_informed_consent:
        immediate = f"{case.title}部署 -> 敏感数据无授权使用"
        near = "用户投诉/隐私监管介入 -> GDPR 调查"
        far = "巨额罚款 + 研究禁令 + 声誉受损"
        leverage = "部署前补全知情同意流程 + 差分隐私"
    elif case.affects_vulnerable_groups and case.potential_harm_severity in ("high", "severe"):
        immediate = f"{case.title}部署 -> 弱势群体承担高风险"
        near = "群体歧视曝光 -> 媒体报道/集体诉讼"
        far = "品牌危机 + 监管介入 + 研究成果被撤"
        leverage = "引入公平性度量(demographic parity) + 人工审核节点"
    elif not case.has_informed_consent:
        immediate = f"{case.title}部署 -> 用户不知情参与研究"
        near = "知情权争议 -> 伦理委员会审查"
        far = "IRB 驳回 + 数据作废 + 重新设计研究"
        leverage = "补充知情同意机制 + 匿名化处理"
    else:
        immediate = f"{case.title}部署 -> 常规 AI 研究风险"
        near = "小幅偏见/性能偏差 -> 内部审计发现"
        far = "迭代优化 + 成果可发表"
        leverage = "定期偏见审计 + 预注册研究方案(preregistration)"
    return {
        "case_id": case.case_id,
        "immediate": immediate,
        "near": near,
        "far": far,
        "leverage_point": leverage,
        "framework": "天道推演: 局势拆解->因果建模->3层推演树->高杠杆点",
    }

print("=== AI 红队测试伦理验证（garak/PyRIT 概念）===")
for case in RESEARCH_CASES:
    rt = redteam_ethics_validation(case)
    print(f"[{rt['case_id']}] {rt['domain']:10s} | garak probes={rt['garak_probes']} | PyRIT 风险={rt['pyrit_risk_score']:5.1f}/100")
print(f"\n  伦理依据: {rt['ethics_basis']}")

print("\n=== 天道推演预判 AI 研究伦理风险路径（3层推演）===")
for case in RESEARCH_CASES:
    td = tian_dao_ethics_projection(case)
    print(f"\n[{td['case_id']}] {td['framework']}")
    print(f"  immediate: {td['immediate']}")
    print(f"  near:      {td['near']}")
    print(f"  far:       {td['far']}")
    print(f"  高杠杆点:  {td['leverage_point']}")


=== AI 红队测试伦理验证（garak/PyRIT 概念）===
[C001] marketing  | garak probes=['dan', 'promptinject', 'goodside'] | PyRIT 风险= 10.0/100
[C002] marketing  | garak probes=['dan', 'promptinject', 'goodside'] | PyRIT 风险= 35.0/100
[C003] marketing  | garak probes=['dan', 'promptinject', 'goodside'] | PyRIT 风险= 80.0/100
[C004] marketing  | garak probes=['dan', 'promptinject', 'goodside'] | PyRIT 风险= 10.0/100
[C005] hr         | garak probes=['dan', 'encoding', 'leakreplay'] | PyRIT 风险= 80.0/100
[C006] finance    | garak probes=['promptinject', 'leakreplay', 'packagehallucination'] | PyRIT 风险= 65.0/100
[C007] security   | garak probes=['encoding', 'promptinject', 'leakreplay'] | PyRIT 风险= 90.0/100
[C008] healthcare | garak probes=['dan', 'snowball', 'goodside'] | PyRIT 风险= 55.0/100

  伦理依据: 红队测试履行 Belmont 善行原则（最大化收益、最小化伤害）

=== 天道推演预判 AI 研究伦理风险路径（3层推演）===

[C001] 天道推演: 局势拆解->因果建模->3层推演树->高杠杆点
  immediate: AI个性化推荐系统用户行为研究部署 -> 常规 AI 研究风险
  near:      小幅偏见/性能偏差 -> 内部审计发现
  far:       迭代优化 + 成果可发表
  高杠杆点: 

## 总结

本 notebook 实现了R6研究伦理与AI治理的完整方法论工具链：

1. **Belmont Report 伦理审查**：三原则6审查项的pydantic schema + IRB评分器
2. **NIST AI RMF 研究伦理映射**：从研究伦理视角（非企业治理视角）映射四步循环
3. **EU AI Act 研究合规**：按真实条款判定风险等级
4. **pandas 伦理热力图**：识别合规短板
5. **AI红队测试伦理验证**：garak/PyRIT 概念作为善行原则的履行
6. **天道推演**：预判AI研究伦理风险路径，识别高杠杆干预点

**与模块R收官的关联**：R6是模块R（博士研究方法论）的收官单元。R1（DSR）定义研究问题、R2（行动研究）设计企业实验、R3（混合方法）设计评估、R4（PRISMA）做文献综述、R5（IMRaD）写论文、**R6（研究伦理）守住底线**。
